In [0]:
from pyspark.sql import functions as F

regions = (
    spark.table("workspace.bank_lakehouse.bronze_regions")
    .select(
        F.col("shape_iso").alias("region_iso"),
        F.col("shape_name").alias("region_name"),
        F.expr("st_geomfromgeojson(geometry_geojson)").alias("region_geom"),
    )
)

agencies = spark.table("workspace.bank_lakehouse.silver_osm_banks") \
    .select("osm_key", "lat", "lon", "bank_name")

joined = agencies.join(
    F.broadcast(regions),
    F.expr("st_contains(region_geom, st_point(lon, lat, 4326))"),
    "left",
)

print("Nombre de régions trouvées par agence :")
joined.groupBy("osm_key").agg(F.count("region_iso").alias("n_regions")) \
    .groupBy("n_regions").count().orderBy("n_regions").show()

display(joined.groupBy("region_iso", "region_name").count().orderBy(F.desc("count")))

In [0]:
from pyspark.sql import functions as F

expected = [
    "banque", "banque participative", "banque centrale", "transfert d'argent",
    "crédit à la consommation", "crédit immobilier", "microcrédit",
    "change", "assurance", "non identifié",
]

display(
    spark.table("workspace.bank_gold.dim_bank")
    .groupBy("institution_type").count()
    .withColumn("dans_la_liste", F.col("institution_type").isin(expected))
    .withColumn("longueur", F.length("institution_type"))
    .withColumn("hex", F.hex("institution_type"))
    .orderBy("dans_la_liste")
)

In [0]:
display(spark.sql("""
    select *, hex(value_field) as hex
    from workspace.bank_gold_dbt_test__audit.accepted_values_dim_bank_5bcf7ea26f7306e3a6898fbcbe676eb5
"""))

In [0]:
display(spark.sql("""
    select region_name, population_2024, bank_branches, bank_branches_per_10k,
           money_transfer_points, microcredit_points
    from workspace.bank_gold.mart_region_kpis
    order by bank_branches_per_10k desc
"""))

In [0]:
display(
    spark.sql("DESCRIBE HISTORY workspace.bank_lakehouse.silver_osm_banks")
    .selectExpr(
        "version", "timestamp", "operation",
        "operationMetrics['numTargetRowsInserted'] as inserees",
        "operationMetrics['numTargetRowsUpdated'] as modifiees",
    )
    .orderBy("version")
)

In [0]:
display(spark.sql("""
    with changed as (
        select osm_key
        from workspace.bank_lakehouse.silver_osm_banks
        where updated_at >= timestamp'2026-11-07 16:50:00'
    )
    select 'avant (v5)' as etat, osm_key, name, brand, operator, bank_name,
           region_name, lat, lon, is_active
    from workspace.bank_lakehouse.silver_osm_banks version as of 5
    where osm_key in (select osm_key from changed)
    union all
    select 'après (v6)', osm_key, name, brand, operator, bank_name,
           region_name, lat, lon, is_active
    from workspace.bank_lakehouse.silver_osm_banks
    where osm_key in (select osm_key from changed)
"""))